# Global Moran's I and LISA

This analysis reads territorial KPIs exclusively from `dw.v_kpi_ageb`. The reference sample is the 483 AGEBs in Mérida's contiguous city core. Queen contiguity is row-standardized; its island is attached to its nearest neighbor. Row-standardized KNN-6 provides the sensitivity check. Every statistic uses 999 permutations and α = 0.05. Spatial association does not establish causation.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from esda import Moran, Moran_Local
from libpysal.weights import lag_spatial
from src.analysis.data import load_kpis
from src.analysis.spatial_weights import build_weights

MAPS = ROOT / 'outputs/maps'
FIGURES = ROOT / 'outputs/figures'
MAPS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## Warehouse prerequisite

Start PostGIS and run the complete ETL before executing this notebook. Loading with `exclude_low_population=False` preserves the prescribed 483-AGEB weights universe. Results for rate indicators must explicitly discuss instability in AGEBs with fewer than 100 residents.

In [ ]:
kpis = load_kpis(city_core_only=True, exclude_low_population=False).set_index('cvegeo')
assert len(kpis) == 483, f'Expected 483 city-core AGEBs; got {len(kpis)}'
INDICATORS = {
    'business_density_km2': 'Business density',
    'crime_rate_per_1k': 'Crime rate per 1,000 residents',
    'pop_density_km2': 'Population density',
    'pea_rate': 'Economically active population rate',
}
display(kpis[list(INDICATORS) + ['low_population']].describe().T)

## Global Moran's I

Each indicator uses its complete observations only, and Queen and KNN-6 are built on the identical indicator-specific sample. The table reports observed I, analytical expectation, permutation z-score, pseudo p-value and sample size.

In [ ]:
results = []
samples = {}
for column, label in INDICATORS.items():
    sample = kpis.loc[kpis[column].notna(), [column, 'geom']].copy()
    if len(sample) < 7 or sample[column].nunique() < 2:
        print(f'Skipped {column}: insufficient variation or observations')
        continue
    samples[column] = sample
    values = sample[column].to_numpy(dtype=float)
    for kind, kind_label in [('queen', 'Queen'), ('knn', 'KNN-6')]:
        weights = build_weights(sample, kind=kind, k=6)
        statistic = Moran(values, weights, permutations=999)
        results.append({
            'indicator': label, 'weights': kind_label, 'n': len(sample),
            'I': statistic.I, 'E[I]': statistic.EI,
            'z_sim': statistic.z_sim, 'p_sim': statistic.p_sim,
            'permutations': statistic.permutations,
        })
moran_results = pd.DataFrame(results)
display(moran_results.round(4))

In [ ]:
for column, sample in samples.items():
    values = sample[column].to_numpy(dtype=float)
    z = (values - values.mean()) / values.std(ddof=0)
    weights = build_weights(sample, kind='queen')
    lag = lag_spatial(weights, z)
    slope = Moran(values, weights, permutations=999).I
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(z, lag, s=16, alpha=.65)
    xline = np.array([z.min(), z.max()])
    ax.plot(xline, slope * xline, color='crimson', label=f'I = {slope:.3f}')
    ax.axhline(0, color='0.6', linewidth=.8); ax.axvline(0, color='0.6', linewidth=.8)
    ax.set(xlabel='Standardized value', ylabel='Spatial lag', title=f"Moran scatterplot: {INDICATORS[column]}")
    ax.legend(); fig.tight_layout()
    fig.savefig(FIGURES / f'32_moran_{column}.png', dpi=180)
    plt.close(fig)

## Local Moran (LISA) clusters

The first two available indicators are mapped. Only locations with permutation p < 0.05 receive a cluster label; all others are shown as not significant. HH and LL indicate similar local values, whereas HL and LH identify spatial outliers.

In [ ]:
CLUSTERS = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
COLORS = {'HH': '#b2182b', 'LL': '#2166ac', 'HL': '#ef8a62', 'LH': '#67a9cf', 'Not significant': '#d9d9d9'}
significant_agebs = []
for column in list(samples)[:2]:
    sample = samples[column].copy()
    weights = build_weights(sample, kind='queen')
    local = Moran_Local(sample[column].to_numpy(dtype=float), weights, permutations=999)
    sample['cluster'] = [CLUSTERS[q] if p < .05 else 'Not significant' for q, p in zip(local.q, local.p_sim)]
    significant_agebs.append(sample.loc[sample['cluster'].ne('Not significant'), [column, 'cluster']].assign(indicator=column))
    fig, ax = plt.subplots(figsize=(8, 8))
    for cluster, color in COLORS.items():
        sample.loc[sample['cluster'].eq(cluster)].plot(ax=ax, color=color, edgecolor='white', linewidth=.15, label=cluster)
    ax.set_title(f"LISA clusters: {INDICATORS[column]} (p < 0.05)")
    ax.set_axis_off(); ax.legend(loc='lower left'); fig.tight_layout()
    fig.savefig(MAPS / f'32_lisa_{column}.png', dpi=180, bbox_inches='tight')
    plt.close(fig)
significant_agebs = pd.concat(significant_agebs).reset_index()
display(significant_agebs.sort_values(['indicator', 'cluster', 'cvegeo']))

## Interpretation guide

A small pseudo p-value indicates that the observed spatial pattern is unusual under random permutation, not that neighboring conditions cause one another. Differences between Queen and KNN-6 show sensitivity to the neighborhood definition. Results are also subject to temporal mismatch between Census 2020 and DENUE 2026, small population denominators, the modifiable areal unit problem, and any limitations of the crime source.